# Day 17 · Light & dark — คืนลีโอนิดส์เหนือประเทศไทย

**#30DayMapChallenge 2026 · 17 พ.ย. 2569** · โจทย์: *"Play with visual effects. What can you map with just light and shadows?"*

คืน 17 → เช้า 18 พ.ย. คือคืนที่ฝนดาวตกกลุ่มลีโอนิดส์ถึงจุดสูงสุด (IMO: 17 พ.ย. 23:45 UT
= 18 พ.ย. 06:45 ICT ซึ่ง *หลังอาทิตย์ขึ้น* ในไทย) แผ่นนี้จึงใช้แสงจริงสองอย่าง
(radiant กับสนธยา/จันทร์) และเงาจริงหนึ่งอย่าง (ปริมาณแผนที่รอบจุดจาก Day 16)

| ต้องรันตามลำดับ | ไฟล์ |
|---|---|
| 1 | `tools/astro17.py` — LST + อาทิตย์ + จันทร์ (Meeus series ที่ตัดพจน์) + หาเวลาขึ้น/ตก |
| 2 | `tools/day17_leonids.py` — คำนวณต่อ 83 จุด, ตรวจเทียบ ephemeris, วาด 3 ภาพ, เขียน artifact |

**สิ่งที่ต้องรู้ก่อนเชื่อตัวเลขในแผ่นนี้**
- ค่าคงที่ (ZHR 15, radiant RA 152°/Dec +22°, peak 23:45 UT) มาจาก IMO Meteor Shower Calendar 2026
- สูตรคือ `ZHR × sin(ความสูง radiant)` — first-order ไม่คิดเมฆ ไม่คิด limiting magnitude
- ephemeris ตรวจแล้ว: อาทิตย์ขึ้นต่างจากตารางที่เผยแพร่ **1.0 นาที** แต่จันทร์ตกต่าง **67 นาที**
  จึงใช้ค่าจันทร์เฉพาะข้อสรุปที่ไม่ก้ำกึ่ง และรายงานทั้งสองค่าไว้เสมอ
- มุม "หันหน้าไปทางไหน" ใช้ 65° จาก radiant เป็น **สมมติฐาน** (ค่านิยมทั่วไป ไม่ใช่ค่าที่วัด)

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 17 · Light & dark — "Play with visual effects. What can you map with just light and shadows?"

ข้อความโจทย์ดึงจาก 30daymapchallenge.com อ่านสดวันที่ 17 พ.ย.

คืนนี้ (17 → เช้า 18 พ.ย.) เป็นคืนที่ฝนดาวตกกลุ่มลีโอนิดส์ถึงจุดสูงสุดตามปฏิทิน IMO
ผมจึงใช้ "แสงกับเงา" สองอย่างที่วัดได้จริงในคืนนั้น
  แสง = แสงที่ฟ้าให้มา (ความสูงของจุดกำเนิดดาวตก + แสงจันทร์ + สนธยา)
  เงา = เงาที่พื้นที่มี (แผนที่รอบจุดนั้นถูกเขียนไว้แค่ไหน — ต่อจาก Day 16)

ทุกตัวเลขในโพสต์อ่านจาก day17_leonids_sites.csv / day17_stats.json เท่านั้น
แหล่งค่าคงที่ดาราศาสตร์: scratch/imo_cal2026.txt (ไฟล์ PDF ของ IMO Meteor Shower Calendar 2026
ที่ดาวน์โหลดและแตกข้อความไว้ใน session นี้) + เวลาที่เผยแพร่สำหรับตรวจเทียบ
"""
import os, sys, json, time, math, datetime as dt
import numpy as np, pandas as pd, geopandas as gpd
from shapely.validation import make_valid
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection
from scipy.stats import spearmanr

WORK = r"D:\GIS30Days"
OUT_EARLY = os.path.join(WORK, "out", "day17")
os.makedirs(OUT_EARLY, exist_ok=True)
sys.path.insert(0, os.path.join(WORK, "tools"))
try:
    import astro17 as AST
    from guards import guard_bbox, guard_crs, guard_variance, guard_labels, lint_thai, hav_km
except ModuleNotFoundError:
    # บน Colab ไม่มีโฟลเดอร์ tools ของเวิร์กสเปซ — ดึงห้องเครื่องมือกับห้องผู้ตัดสินจาก repo มาใช้
    import urllib.request
    RAW = "https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/tools/"
    for stem in ("astro17.py", "guards.py"):
        dst = os.path.join(OUT_EARLY, stem)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(RAW + stem, dst)
    sys.path.insert(0, OUT_EARLY)
    import astro17 as AST
    from guards import guard_bbox, guard_crs, guard_variance, guard_labels, lint_thai, hav_km

T0 = time.time()
WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day17")
os.makedirs(OUT, exist_ok=True)
for _f in sorted(os.listdir(CACHE)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day17_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

TXT = []
def T(s):
    TXT.append(s); return s

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK, MUTED = "#0d1522", "#e8e3d3", "#9fb2c8"
ACCENT, WARM, COOL, TEAL = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf"
ICT = dt.timezone(dt.timedelta(hours=7))
NIGHT_START = dt.datetime(2026, 11, 17, 18, 0, tzinfo=ICT)     # 17 พ.ย. 18:00 ICT
NIGHT_END = dt.datetime(2026, 11, 18, 8, 0, tzinfo=ICT)        # 18 พ.ย. 08:00 ICT
BEST_HOUR = dt.datetime(2026, 11, 18, 3, 0, tzinfo=ICT)        # ชั่วโมงที่วัดส่วนใหญ่ใช้เปรียบเทียบ
LOOK_DEG = 65.0                                                # มุมห่างจาก radiant ที่แนะนำ (สมมติฐาน — ป้ายไว้)
j_start = AST.jd_utc(2026, 11, 17, 11)     # 18:00 ICT = 11:00 UTC
j_end = AST.jd_utc(2026, 11, 18, 1)        # 08:00 ICT = 01:00 UTC
j_best = AST.jd_utc(2026, 11, 17, 20)      # 03:00 ICT ของวันที่ 18 = 20:00 UTC ของวันที่ 17

## ตอน 1) ค่าคงที่ + ตรวจเทียบ

In [ ]:
# ---------------------------------------------------------------- 1. ค่าคงที่ + ตรวจเทียบ
CONST = dict(shower="013 LEO (Leonids)", zhr=AST.ZHR, radiant_ra_deg=AST.RADIANT[0],
             radiant_dec_deg=AST.RADIANT[1], peak_utc="2026-11-17 23:45 UT",
             peak_ict="2026-11-18 06:45 ICT",
             source="IMO Meteor Shower Calendar 2026 (ไฟล์ PDF ดาวน์โหลดมาแตกข้อความใน session นี้)",
             velocity_km_s=71.0, parent_comet="55P/Tempel-Tuttle")
log(f"ค่าคงที่ IMO: ZHR {CONST['zhr']} · radiant α={CONST['radiant_ra_deg']}° δ={CONST['radiant_dec_deg']}° · "
    f"peak {CONST['peak_utc']} = {CONST['peak_ict']}")

BKK = (13.7543, 100.5056)
sun_up, sun_dn = AST.rise_set("sun", *BKK, j_start, j_end, -0.833)
_, moon_dn = AST.rise_set("moon", *BKK, j_start, j_end, -0.125)
ast_up, ast_dn = AST.rise_set("sun", *BKK, j_start, j_end, -18.0)
rad_up, rad_dn = AST.rise_set("radiant", *BKK, j_start, j_end, 0.0)
fmt = lambda j: None if j is None else (dt.datetime.fromtimestamp((j - 2440587.5) * 86400, dt.timezone.utc)
                                        + dt.timedelta(hours=7))
iso = lambda j: None if j is None else fmt(j).strftime("%Y-%m-%d %H:%M")
PUBLISHED = dict(bangkok_sunrise_ict="2026-11-18 06:19", bangkok_moonset_ict="2026-11-17 23:19",
                 source="timebie.com moon table for Bangkok (November) — หน้าตารางที่อ่านสด 21:20 ICT")
VAL = dict(sunrise_model=iso(sun_up), sunrise_published=PUBLISHED["bangkok_sunrise_ict"],
           moonset_model=iso(moon_dn), moonset_published=PUBLISHED["bangkok_moonset_ict"],
           radiant_rise_model=iso(rad_up),
           astro_dark_end_model=iso(ast_up),
           moon_illumination_at_best_hour=round(100 * AST.moon_ra_dec(j_best)[3], 1),
           moon_elongation_at_best_hour=round(AST.moon_ra_dec(j_best)[2], 1))
def _diff_min(a, b):
    if a is None or b is None:
        return None
    ta = dt.datetime.fromisoformat(a)
    tb = dt.datetime.fromisoformat(b)
    return round((ta - tb).total_seconds() / 60.0, 1)
VAL["sunrise_diff_min"] = _diff_min(VAL["sunrise_model"], VAL["sunrise_published"])
VAL["moonset_diff_min"] = _diff_min(VAL["moonset_model"], VAL["moonset_published"])
log(f"ตรวจเทียบ: อาทิตย์ขึ้น {VAL['sunrise_model']} vs เผยแพร่ {VAL['sunrise_published']} "
    f"(ต่าง {VAL['sunrise_diff_min']} นาที) · จันทร์ตก {VAL['moonset_model']} vs {VAL['moonset_published']} "
    f"(ต่าง {VAL['moonset_diff_min']} นาที) · แสงจันทร์สว่าง {VAL['moon_illumination_at_best_hour']}% "
    f"ที่ 03:00 ICT แต่ตกไปแล้ว")

## ตอน 2) ข้อมูลซีรีส์ + ทะเบียน

In [ ]:
# ---------------------------------------------------------------- 2. ข้อมูลซีรีส์ + ทะเบียน
df = pd.read_csv(os.path.join(REPO, "data/darksky_thailand.csv"))
df["name"] = df["name"].astype(str).str.strip()
assert len(df) == 83, f"ทะเบียนต้องมี 83 จุด ได้ {len(df)}"
SITES = np.column_stack([df.longitude.values, df.latitude.values])
guard_bbox(SITES, 96.5, 107.6, 4.3, 21.7, "darksky-sites")
ne = gpd.read_file(os.path.join(CACHE, "ne_10m_admin_0_countries.shp"))
TH = make_valid(ne[ne.NAME == "Thailand"].geometry.iloc[0])
PARTS = list(TH.geoms)
guard_crs(TH.bounds, "thailand-outline", expect="degrees")
d16 = pd.read_csv(os.path.join(REPO, "posts/day16-collaborative-map/day16_sites_osm.csv"))
assert len(d16) == 83, "ต้องต่อจาก Day 16 (83 จุด) ได้"
d11 = json.load(open(os.path.join(REPO, "posts/day11-sound/day11_hex_quadrants.geojson"), encoding="utf-8"))
noise_by_cell = {}
for f in d11["features"]:
    c = f["properties"].get("cell")
    if c is not None:
        noise_by_cell[str(c)] = float(f["properties"].get("noise_km", np.nan))
import h3

## ตอน 3) คำนวณต่อจุดของคืน 17 → 18 พ.ย.

In [ ]:
# ------------------------------------------------- 3. คำนวณต่อจุดของคืน 17 → 18 พ.ย.
def sky_at(lat, lon, jd):
    ra, dec, _, _ = AST.moon_ra_dec(jd)
    sra, sdec = AST.sun_ra_dec(jd)
    return (AST.icrs_alt(AST.RADIANT[0], AST.RADIANT[1], lat, lon, jd),
            AST.icrs_alt(ra, dec, lat, lon, jd),
            AST.icrs_alt(sra, sdec, lat, lon, jd))

def look_point(lat, lon, jd, sep_deg=LOOK_DEG):
    """จุดบนท้องฟ้าที่ห่างจาก radiant เป็นมุม sep_deg บนวงกลมใหญ่ที่ลากไปทาง zenith
    (คำแนะนำของ Moissl ใน BBC: 'ให้ radiant อยู่ด้านหลัง' — ผมแปลงเป็นมุมแน่นอนเพื่อวาดได้)"""
    ra, dec = AST.RADIANT
    # ทิศหน่วยในพิกัด equatorial
    def vec(ra_, dec_):
        r, d = math.radians(ra_), math.radians(dec_)
        return np.array([math.cos(d) * math.cos(r), math.cos(d) * math.sin(r), math.sin(d)])
    v_r = vec(ra, dec)
    # zenith ใน equatorial = (lat, lst)
    T = (jd - 2451545.0) / 36525.0
    gmst = (280.46061837 + 360.98564736629 * (jd - 2451545.0) + 0.000387933 * T * T) % 360.0
    lst = (gmst + lon) % 360.0
    v_z = vec(lst, lat)
    if np.linalg.norm(v_z - v_r) < 1e-9:
        return None
    e = (v_z - np.dot(v_z, v_r) * v_r)
    e /= np.linalg.norm(e)
    s, c = math.sin(math.radians(sep_deg)), math.cos(math.radians(sep_deg))
    v_t = c * v_r + s * e
    dec_t = math.degrees(math.asin(max(-1, min(1, v_t[2]))))
    ra_t = math.degrees(math.atan2(v_t[1], v_t[0])) % 360.0
    H = math.radians((lst - ra_t + 180) % 360 - 180)
    la = math.radians(lat)
    alt = math.degrees(math.asin(math.sin(math.radians(dec_t)) * math.sin(la) +
                                 math.cos(math.radians(dec_t)) * math.cos(la) * math.cos(H)))
    az = math.degrees(math.atan2(math.sin(H), math.cos(H) * math.sin(la) -
                                 math.tan(math.radians(dec_t)) * math.cos(la))) % 360.0
    return dict(alt=round(alt, 2), az=round(az, 2), sep_deg=sep_deg)

rows = []
for i, r in df.iterrows():
    lat, lon = float(r.latitude), float(r.longitude)
    h_rad, h_moon, h_sun = sky_at(lat, lon, j_best)
    rate = AST.ZHR * math.sin(math.radians(h_rad)) if h_rad > 0 else 0.0
    # หน้าต่างที่ "มืดจริง + ไม่มีจันทร์" ของจุดนี้
    dark = []
    for k in range(int((j_end - j_start) * 1440 / 5.0)):
        jd = j_start + k * 5.0 / 1440.0
        a_r, a_m, a_s = sky_at(lat, lon, jd)
        dark.append((jd, a_r, a_m, a_s))
    good = [d for d in dark if d[3] < -18.0 and d[2] < -0.125 and d[1] > 0]
    win_min = len(good) * 5
    lp = look_point(lat, lon, j_best)
    cell = h3.latlng_to_cell(lat, lon, 5)
    noise = noise_by_cell.get(str(cell), np.nan)
    rows.append(dict(idx=i, name=r["name"], province=str(r.province), holder_type=str(r.type),
                     lat=round(lat, 4), lon=round(lon, 4),
                     radiant_alt_0300=round(h_rad, 2), moon_alt_0300=round(h_moon, 2),
                     sun_alt_0300=round(h_sun, 2), expected_rate_0300=round(rate, 2),
                     dark_moonfree_window_min=win_min,
                     look_az_0300=None if lp is None else lp["az"],
                     look_alt_0300=None if lp is None else lp["alt"],
                     sqm=(None if pd.isna(getattr(r, "sky_quality_sqm", np.nan))
                          else float(r.sky_quality_sqm)),
                     ways_day16=int(d16.ways.iloc[i]), users_day16=int(d16.users.iloc[i]),
                     noise_km_day11=(None if pd.isna(noise) else round(noise, 2))))
S = pd.DataFrame(rows)
assert len(S) == 83
assert (S.moon_alt_0300.values < 0).all(), "ที่ 03:00 ICT จันทร์ต้องต่ำกว่าขอบฟ้าทุกจุด (ไม่งั้นข้อสรุปแสงจันทร์ใช้ไม่ได้)"
assert (S.sun_alt_0300.values < -18).all(), "ที่ 03:00 ICT ต้องอยู่ใน darkness เชิงดาราศาสตร์ทุกจุด"
guard_variance(S.radiant_alt_0300.values, "radiant-alt", min_spread_km=1.0)
guard_variance(S.expected_rate_0300.values, "expected-rate", min_spread_km=0)
log(f"radiant ที่ 03:00 ICT: ต่ำสุด {S.radiant_alt_0300.min():.1f}° · มัธยฐาน {S.radiant_alt_0300.median():.1f}° · "
    f"สูงสุด {S.radiant_alt_0300.max():.1f}° · อัตราคาดหมาย {S.expected_rate_0300.min():.1f}–"
    f"{S.expected_rate_0300.max():.1f} ดวง/ชม.")
log(f"หน้าต่างมืด+ไร้จันทร์: สั้นสุด {S.dark_moonfree_window_min.min()} นาที · "
    f"มัธยฐาน {S.dark_moonfree_window_min.median():.0f} · ยาวสุด {S.dark_moonfree_window_min.max()} นาที")

rho_r, p_r = spearmanr(S.expected_rate_0300.values, S.ways_day16.values)
rho_s, p_s = spearmanr(S.sqm.values, S.expected_rate_0300.values)
AGG = dict(sites=int(len(S)),
           radiant_alt_min=round(float(S.radiant_alt_0300.min()), 2),
           radiant_alt_median=round(float(S.radiant_alt_0300.median()), 2),
           radiant_alt_max=round(float(S.radiant_alt_0300.max()), 2),
           rate_min=round(float(S.expected_rate_0300.min()), 2),
           rate_median=round(float(S.expected_rate_0300.median()), 2),
           rate_max=round(float(S.expected_rate_0300.max()), 2),
           window_min=int(S.dark_moonfree_window_min.min()),
           window_median=float(S.dark_moonfree_window_min.median()),
           window_max=int(S.dark_moonfree_window_min.max()),
           sites_window_zero=int((S.dark_moonfree_window_min == 0).sum()),
           spearman_rate_vs_ways=round(float(rho_r), 3), spearman_p_rate_ways=float(p_r),
           spearman_sqm_vs_rate=round(float(rho_s), 3), spearman_p_sqm_rate=float(p_s),
           sqm_min=float(np.nanmin(S.sqm.values)), sqm_median=float(np.nanmedian(S.sqm.values)),
           sqm_max=float(np.nanmax(S.sqm.values)), sqm_missing=int(S.sqm.isna().sum()),
           rate_max_over_min=round(float(AGG_RATE_MAX / AGG_RATE_MIN), 3) if False else
                             round(float(S.expected_rate_0300.max() / S.expected_rate_0300.min()), 3),
           culmination_deg=round(90 - abs(19.1810 - AST.RADIANT[1]), 2),
           culmination_min_sites=round(float((90 - np.abs(S.lat.values - AST.RADIANT[1])).min()), 2),
           culmination_max_sites=round(float((90 - np.abs(S.lat.values - AST.RADIANT[1])).max()), 2),
           ways_median_top20=float(S.sort_values("expected_rate_0300", ascending=False)
                                   .head(20).ways_day16.median()),
           ways_median_bottom20=float(S.sort_values("expected_rate_0300").head(20).ways_day16.median()))
log(f"อัตราดาวตก ↔ จำนวน way ที่เขียนแผนที่: Spearman rho = {rho_r:.3f} (p = {p_r:.2g}) · "
    f"way มัธยฐานของกลุ่มฟ้าดี 20 อันดับแรก {AGG['ways_median_top20']:.0f} "
    f"เทียบกลุ่มอันดับล่าง 20 {AGG['ways_median_bottom20']:.0f}")

# ============================================================ ภาพ 1 — แสงและเงาของคืนนั้น
def draw_light_shadow():
    fig = plt.figure(figsize=(15.2, 8.4), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.02, 0.10, 0.68, 0.80]); ax.set_facecolor("#05080f")
    ax.add_collection(PolyCollection([list(p.exterior.coords) for p in PARTS],
                                     facecolor="#101a2c", edgecolor="#4a6288", lw=0.9))
    # แสง: halo ความสว่าง = อัตราดาวตกที่คาด
    for _, r in S.iterrows():
        w = max(r.expected_rate_0300, 0.05)
        rad = 0.07 + 0.42 * (w / AGG["rate_max"])
        g = ax.scatter([r.lon], [r.lat], s=(rad * 105) ** 2 / 12, c=[[1, 0.87, 0.5, min(0.92, 0.18 + 0.7 * w / AGG["rate_max"])]],
                       edgecolor="none", zorder=4)
        ax.scatter([r.lon], [r.lat], s=12, c="#fff6df", edgecolor="none", zorder=6)
    # เงา: วงดำ = แผนที่ตรงนั้นแทบไม่มีใครเขียน (ต่อจาก Day 16)
    for _, r in S.iterrows():
        shade = min(1.0, (50 - min(r.ways_day16, 50)) / 50)
        if shade > 0.05:
            ax.scatter([r.lon], [r.lat], s=170 * shade, facecolor="none",
                       edgecolor="#000000", lw=0.8 + 2.2 * shade, zorder=5, alpha=0.85)
    ax.set_xlim(97.2, 106.6); ax.set_ylim(5.2, 20.9); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(T("จุดที่ฟ้าเปิดที่สุดในคืนนี้ เทียบกับจุดที่แผนที่ถูกเขียนน้อยที่สุด"),
                 color=INK, fontsize=12.5, pad=10)
    txt = fig.add_axes([0.72, 0.10, 0.27, 0.80]); txt.axis("off")
    lines = [(T("จุดกำเนิดดาวตก (radiant)"), f"RA {CONST['radiant_ra_deg']}° · Dec +{CONST['radiant_dec_deg']}°"),
             (T("จุดสูงสุด IMO"), CONST["peak_ict"]),
             (T("ZHR"), f"{CONST['zhr']} ดวง/ชม."),
             (T("radiant ที่ 03:00 · ต่ำสุด"), f"{AGG['radiant_alt_min']:.1f}°"),
             (T("radiant ที่ 03:00 · มัธยฐาน"), f"{AGG['radiant_alt_median']:.1f}°"),
             (T("radiant ที่ 03:00 · สูงสุด"), f"{AGG['radiant_alt_max']:.1f}°"),
             (T("อัตราคาดหมาย · มัธยฐาน"), f"{AGG['rate_median']:.1f} ดวง/ชม."),
             (T("อัตราคาดหมาย · สูงสุด"), f"{AGG['rate_max']:.1f} ดวง/ชม."),
             ("", ""),
             (T("จันทร์สว่าง (03:00)"), f"{VAL['moon_illumination_at_best_hour']:.0f}%"),
             (T("แต่จันทร์ตกตั้งแต่"), str(VAL["moonset_model"])[11:16] + " ICT"),
             (T("หน้าต่างมืดไร้จันทร์ · สั้นสุด"), f"{AGG['window_min']} นาที"),
             (T("หน้าต่างมืดไร้จันทร์ · มัธยฐาน"), f"{AGG['window_median']:.0f} นาที"),
             (T("หน้าต่างมืดไร้จันทร์ · ยาวสุด"), f"{AGG['window_max']} นาที"),
             ("", ""),
             (T("Spearman อัตราดาวตกกับ way (Day 16)"), f"{AGG['spearman_rate_vs_ways']:.3f}"),
             (T("way มัธยฐาน 20 จุดฟ้าดีสุด"), f"{AGG['ways_median_top20']:.0f}"),
             (T("way มัธยฐาน 20 จุดฟ้าแย่สุด"), f"{AGG['ways_median_bottom20']:.0f}")]
    y = 0.98
    for k, v in lines:
        if not k:
            y -= 0.03; continue
        txt.text(0.0, y, k, color=MUTED, fontsize=10.2, va="top")
        txt.text(1.0, y, v, color=ACCENT, fontsize=10.2, va="top", ha="right")
        y -= 0.055
    fig.text(0.02, 0.075, T("แสง = halo สีเหลือง ความสว่างตามจำนวนดาวตกที่คาดต่อชั่วโมงที่ 03:00 น." + chr(10) +
                            "เงา = วงดำ ขนาดตามปริมาณแผนที่ที่ *ไม่มีใครเขียน* รอบจุด (ต่อจาก Day 16)" + chr(10) +
                            "แผ่นนี้ใช้แค่สองอย่างนั้น ไม่มีการระบายสีตามหมวดข้อมูลเลย"),
             color=MUTED, fontsize=10, va="top", linespacing=1.6)
    fig.suptitle(T("คืนลีโอนิดส์ 17 → 18 พ.ย. 2569 · แสงจากฟ้า กับเงาจากแผนที่"),
                 color=INK, fontsize=15.5, y=0.985)
    p = os.path.join(OUT, "day17_light_dark.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

# ============================================================ ภาพ 2 — เส้นแสงของคืน
def draw_night():
    lat, lon = 18.7883, 98.9853          # เชียงใหม่ (จุดอ้างอิงของเส้นนี้)
    ts, hr, hm, hs = [], [], [], []
    for k in range(int((j_end - j_start) * 144)):
        jd = j_start + k / 144.0
        a_r, a_m, a_s = sky_at(lat, lon, jd)
        t = dt.datetime.fromtimestamp((jd - 2440587.5) * 86400, dt.timezone.utc) + dt.timedelta(hours=7)
        ts.append(t); hr.append(a_r); hm.append(a_m); hs.append(a_s)
    fig, ax = plt.subplots(figsize=(14.4, 6.4), dpi=150)
    fig.patch.set_facecolor(BG); ax.set_facecolor("#05080f")
    ax.fill_between(ts, -90, 0, color="#02040a")
    ax.fill_between(ts, -18, 0, color="#0a1226")
    ax.fill_between(ts, -12, 0, color="#16223a")
    ax.fill_between(ts, -0.833, 0, color="#24344d")
    ax.plot(ts, hs, color=MUTED, lw=1.6, ls="--", label=T("อาทิตย์"))
    ax.plot(ts, hm, color=COOL, lw=2.0, label=T("จันทร์"))
    ax.plot(ts, hr, color=ACCENT, lw=2.6, label=T("จุดกำเนิดดาวตก (radiant)"))
    ax.axhline(0, color=INK, lw=0.8)
    ax.axhline(-18, color=WARM, lw=0.8, ls=":")
    ax.text(ts[2], -18.9, T("ขอบ darkness เชิงดาราศาสตร์ −18°"), color=WARM, fontsize=9.5)
    pk = dt.datetime(2026, 11, 18, 6, 45, tzinfo=ICT)
    ax.axvline(pk, color=WARM, lw=1.4, ls="--")
    ax.annotate(T("จุดสูงสุด IMO\n06:45 ICT (หลังอาทิตย์ขึ้น)"), (pk, 55), color=WARM, fontsize=10,
                xytext=(-14, 0), textcoords="offset points", ha="right")
    ax.annotate(T("จันทร์ตก " + (fmt(moon_dn).strftime("%H:%M") if fmt(moon_dn) else "-")),
                (fmt(moon_dn), -22), color=COOL, fontsize=10, xytext=(-6, -16),
                textcoords="offset points", ha="right")
    ax.annotate(T("radiant ขึ้น " + (fmt(rad_up).strftime("%H:%M") if fmt(rad_up) else "-")),
                (fmt(rad_up), -40), color=ACCENT, fontsize=10, xytext=(8, 0),
                textcoords="offset points", ha="left")
    ax.set_ylim(-90, 92); ax.set_ylabel(T("ความสูงเหนือขอบฟ้า (องศา)"), color=INK)
    ax.set_title(T(f"เชียงใหม่ · คืน 17 → เช้า 18 พ.ย. 2569 · "
                   f"หน้าต่างที่มืดจริงและไม่มีแสงจันทร์ = {AGG['window_median']:.0f} นาที (มัธยฐาน 83 จุด)"),
                 color=INK, fontsize=13)
    ax.tick_params(colors=INK, labelsize=9.5)
    for s in ax.spines.values(): s.set_color("#33415e")
    ax.legend(facecolor="#05080f", edgecolor="#33415e", labelcolor=INK, fontsize=10, loc="upper left")
    fig.autofmt_xdate()
    fig.text(0.5, 0.012, T("แถบสีเข้มคือระดับความมืดของท้องฟ้า (ดำสนิท = −18° ลงไป) "
                           "ส่วนที่เส้น accent อยู่เหนือ 0° คือช่วงที่เห็นดาวตกได้"),
             color=MUTED, fontsize=10, ha="center")
    fig.suptitle(T("แสงเดียวที่แข่งกับดาวตกในคืนนี้คือแสงสนธยา ไม่ใช่แสงจันทร์"),
                 color=INK, fontsize=15, y=0.98)
    p = os.path.join(OUT, "day17_night_curve.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

# ============================================================ ภาพ 3 -> อันดับจุด + หันหน้าไปทางไหน
def draw_ranking():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15.2, 6.6), dpi=150,
                                  gridspec_kw=dict(width_ratios=[1.15, 1]))
    fig.patch.set_facecolor(BG)
    top = S.sort_values("expected_rate_0300", ascending=False).head(18).iloc[::-1]
    ax.set_facecolor("#05080f")
    ax.barh(np.arange(len(top)), top.expected_rate_0300.values, color=ACCENT,
            edgecolor=BG, lw=0.3, height=0.62)
    ax.set_yticks(np.arange(len(top)))
    ax.set_yticklabels([f"{n[:26]} · {p}" for n, p in zip(top.name.values, top.province.values)],
                       color=INK, fontsize=9.2)
    for i, (v, w) in enumerate(zip(top.expected_rate_0300.values, top.ways_day16.values)):
        ax.text(v + 0.15, i, f"{v:.1f}", va="center", color=INK, fontsize=9)
        ax.text(0.15, i, f"way {w}", va="center", color=MUTED, fontsize=8.2)
    ax.set_xlim(0, AGG["rate_max"] * 1.18)
    ax.set_xlabel(T("ดาวตกที่คาด (ดวง/ชม.) ที่ 03:00 ICT · สูตร ZHR × sin(ความสูง radiant)"))
    ax.set_title(T("18 จุดทะเบียนที่ฟ้าเปิดที่สุดในคืนนี้ (พร้อมตัวเลขว่ารอบจุดมีแผนที่กี่ชิ้น)"),
                 color=INK, fontsize=12)
    ax.tick_params(colors=INK, labelsize=9)
    ax.grid(color="#26364f", lw=0.5, alpha=0.5, axis="x")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color("#33415e")

    ax2.set_facecolor("#05080f")
    sc = ax2.scatter(S.expected_rate_0300.values, S.ways_day16.values,
                     s=52, c=S.sqm.values, cmap="viridis",
                     edgecolor=INK, lw=0.4)
    ax2.set_yscale("log")
    ax2.set_xlabel(T("ดาวตกที่คาด (ดวง/ชม.) ที่ 03:00 ICT"))
    ax2.set_ylabel(T("จำนวน way ในแผนที่รอบจุด (Day 16, log)"))
    ax2.set_title(T(f"ฟ้าสว่างขึ้น แต่แผนที่ไม่ได้สว่างตาม\nSpearman rho = {AGG['spearman_rate_vs_ways']:.3f} "
                    f"(p = {AGG['spearman_p_rate_ways']:.2g})"), color=INK, fontsize=12)
    ax2.tick_params(colors=INK, labelsize=9.5)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.5)
    for s in ax2.spines.values(): s.set_color("#33415e")
    cb = fig.add_axes([0.955, 0.16, 0.012, 0.30])
    plt.colorbar(sc, cax=cb).set_label(T("คุณภาพฟ้า SQM (mag/arcsec²)"), color=INK, fontsize=9)
    cb.tick_params(colors=INK, labelsize=8)
    med = S.sort_values("expected_rate_0300").iloc[len(S) // 2]
    fig.text(0.5, 0.015, T(f"จุดมัธยฐานของคืนนี้ ({med['name'][:24]}): หันหน้าไปทิศ "
                          f"{med.look_az_0300:.0f}° ที่มุมสูง {med.look_alt_0300:.0f}° "
                          f"(ห่างจาก radiant {LOOK_DEG:.0f}° ตามคำแนะนำ 'ให้ radiant อยู่ด้านหลัง')"),
             color=MUTED, fontsize=10.2, ha="center")
    fig.suptitle(T("เงาของแผนที่กับแสงของท้องฟ้า"), color=INK, fontsize=15, y=0.98)
    p = os.path.join(OUT, "day17_ranking.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

P1, P2, P3 = draw_light_shadow(), draw_night(), draw_ranking()

## ตอน 4) artifact

In [ ]:
# ---------------------------------------------------------------- 4. artifact
S.round(3).to_csv(os.path.join(OUT, "day17_leonids_sites.csv"), index=False)
STATS = dict(
    theme="Play with visual effects. What can you map with just light and shadows?",
    theme_source="30daymapchallenge.com หัวข้อ Day 17 'Light & dark' (อ่านสดวันที่ 17 พ.ย.)",
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    constants=CONST,
    validation=VAL,
    published_reference=PUBLISHED,
    aggregates=AGG,
    per_site=[{k: (None if (isinstance(v, float) and pd.isna(v)) else v) for k, v in r.items()}
              for r in S.to_dict("records")],
    top15=[{"name": r["name"], "province": r["province"], "rate": round(float(r["expected_rate_0300"]), 2),
            "radiant_alt": round(float(r["radiant_alt_0300"]), 1), "ways_day16": int(r["ways_day16"]),
            "sqm": (None if pd.isna(r["sqm"]) else float(r["sqm"]))}
           for _, r in S.sort_values("expected_rate_0300", ascending=False).head(15).iterrows()],
    cross_check_day16=dict(ways_field="day16_sites_osm.csv ways", users_field="day16_sites_osm.csv users",
                           sites_matched=int(len(d16)), note="ต่อจาก Day 16 โดยตรง ไม่คำนวณซ้ำ"),
    cross_check_day11=dict(noise_field="day11_hex_quadrants.geojson → ฟิลด์ cell",
                           matched=int(S.noise_km_day11.notna().sum()),
                           note="geojson ของ Day 11 ไม่มีฟิลด์ cell ให้จับคู่ การ join จึงได้ 0 "
                                "และไม่มีตัวเลข noise_km ถูกใช้ในโพสต์นี้เลย (เก็บไว้เป็นหลักฐานว่าลองแล้วไม่ผ่าน)"),
    register_quality_field=dict(column="sky_quality_sqm", min=AGG["sqm_min"], median=AGG["sqm_median"],
                                max=AGG["sqm_max"], missing=AGG["sqm_missing"],
                                note="ทะเบียนบันทึกเป็นจำนวนเต็ม mag/arcsec² การเปรียบเทียบระหว่างจุดจึงหยาบ"),
    assumptions=dict(
        zenith_formula=dict(expression="มุมสูงสุดของวัตถุ = 90 − |ละติจูด − declination|",
                            constant_deg=90, note="ค่า 90 ในโพสต์มาจากสูตรนี้ ไม่ใช่ตัวเลขที่วัด"),
        rate_formula="จำนวนดาวตกที่คาด = ZHR × sin(ความสูงของ radiant) — first-order correction "
                     "ที่ IMO/หนังสือดาวตกใช้ ไม่คิดเรื่อง limiting magnitude, เมฆ หรือ extinction ของบรรยากาศ",
        look_direction=f"จุดที่แนะนำให้หันหน้าไป ห่างจาก radiant {LOOK_DEG:.0f}° บนวงกลมใหญ่ที่ลากไปทาง "
                       "zenith (แปลงคำแนะนำ 'ให้ radiant อยู่ด้านหลัง' ของ Moissl ใน BBC เป็นตัวเลข) "
                       "มุม 65° เป็นค่าประมาณที่ใช้กันทั่วไป ไม่ใช่ค่าที่วัดจากข้อมูล",
        moonset_uncertainty="แบบจำลองจันทร์ให้ค่าต่างจากตารางที่เผยแพร่ราว 60 นาที "
                            "(ดู validation) ข้อสรุป 'ไม่มีแสงจันทร์ช่วง 02:00–05:00' ยังอยู่ได้ "
                            "เพราะจันทร์ตกก่อนหน้านั้นมาก"),
    figures=["day17_light_dark.png", "day17_night_curve.png", "day17_ranking.png"],
    metric_definitions={
        "radiant_alt_0300": "ความสูงเหนือขอบฟ้า (องศา) ของจุดกำเนิดดาวตก ณ 03:00 ICT ของวันที่ 18 พ.ย. คำนวณจาก α/δ ของ IMO กับเวลาท้องถิ่นของแต่ละจุด",
        "expected_rate_0300": "ZHR × sin(radiant_alt_0300) = จำนวนดาวตกต่อชั่วโมงที่คาดสำหรับผู้สังเกตสายตาดี under dark sky (สูตรใน assumptions)",
        "moon_alt_0300": "ความสูงของจันทร์ ณ เวลาเดียวกัน (ติดลบทุกจุด = ใต้ขอบฟ้า)",
        "sun_alt_0300": "ความสูงของอาทิตย์ ณ เวลาเดียวกัน (ติดลบเกิน 18° = darkness เชิงดาราศาสตร์)",
        "dark_moonfree_window_min": "จำนวนนาทีในคืน 17→18 พ.ย. ที่อาทิตย์ต่ำกว่า −18° *และ* จันทร์ต่ำกว่าขอบฟ้า *และ* radiant สูงกว่า 0° (สุ่มตัวอย่างทุก 5 นาที)",
        "look_az_0300 / look_alt_0300": "ทิศ (มุมจากเหนือตามเข็มนาฬิกา) และมุมสูงของจุดที่แนะนำให้หันหน้าไป ณ 03:00 ICT",
        "ways_day16": "จำนวน way ในรัศมี 1 กม. รอบจุดจาก Day 16 (ใช้ต่อโดยตรง)",
        "sqm": "ค่าคุณภาพท้องฟ้า (mag/arcsec²) จากฟิลด์ sky_quality_sqm ของทะเบียน",
        "rate_max_over_min": "อัตราคาดหมายสูงสุด ÷ ต่ำสุด (เท่า) — บอกว่าทั้งประเทศต่างกันน้อยแค่ไหน",
        "culmination_deg": "มุมสูงสุดที่ radiant จะขึ้นถึง = 90 − |ละติจูด − Dec| คำนวณที่ละติจูดพะเยา 19.181°",
        "culmination_min_sites / culmination_max_sites": "ค่าต่ำสุด/สูงสุดของ 90 − |ละติจูดจุด − Dec| ในบรรดา 83 จุด",
        "sqm": "ค่าคุณภาพท้องฟ้า mag/arcsec² จากฟิลด์ sky_quality_sqm ของทะเบียน (ข้อมูลบันทึกเป็นจำนวนเต็ม จึงละเอียดแค่ 1 ขั้น)",
        "noise_km_day11": "ระยะถึงแหล่งเสียงที่ใกล้ที่สุด (กม.) จาก artifact Day 11"},
    note="ค่าคงที่ดาราศาสตร์มาจาก IMO Meteor Shower Calendar 2026 (PDF ที่ดาวน์โหลดมาแตกข้อความ) "
         "ส่วนเวลาขึ้น/ตกคำนวณด้วย ephemeris ใน tools/astro17.py ซึ่งตรวจเทียบกับตารางที่เผยแพร่แล้ว")
json.dump(STATS, open(os.path.join(OUT, "day17_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=lambda o: (o.item() if hasattr(o, "item")
                                                           else str(o) if isinstance(o, (dt.datetime, dt.date)) else str(o)))
log("เขียน day17_stats.json + day17_leonids_sites.csv")

## ตอน 5) referee

In [ ]:
# ---------------------------------------------------------------- 5. referee
for f in ["day17_light_dark.png", "day17_night_curve.png", "day17_ranking.png",
          "day17_leonids_sites.csv", "day17_stats.json"]:
    assert os.path.getsize(os.path.join(OUT, f)) > 0, f"ไฟล์ว่าง: {f}"
json.load(open(os.path.join(OUT, "day17_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day17-texts")
assert AGG["rate_max"] <= AST.ZHR, "อัตราคาดหมายต้องไม่เกิน ZHR"
assert (S.moon_alt_0300.values < -0.125).all()
_src = os.path.abspath(__file__) if "__file__" in globals() else None
if _src and os.path.exists(_src):
    _bad = []
    for _n, _l in enumerate(open(_src, encoding="utf-8").read().split(chr(10)), 1):
        if any(0x0E00 <= ord(c) <= 0x0E7F for c in _l):
            for c in _l:
                o = ord(c)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0xAC00 <= o <= 0xD7AF) or (0x4E00 <= o <= 0x9FFF):
                    _bad.append((_n, c))
    assert not _bad, f"พบอักษรไม่ใช่ไทยในบรรทัดไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว")
log("สรุป: " + json.dumps(dict(radiant_median=AGG["radiant_alt_median"], rate_median=AGG["rate_median"],
                               rate_max=AGG["rate_max"], window_median=AGG["window_median"],
                               moonset_model=VAL["moonset_model"], moonset_published=VAL["moonset_published"],
                               moonset_diff_min=VAL["moonset_diff_min"],
                               rho_rate_ways=AGG["spearman_rate_vs_ways"],
                               ways_top20=AGG["ways_median_top20"], ways_bottom20=AGG["ways_median_bottom20"]),
                          ensure_ascii=False))
print("DONE day17 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()